For 1 item/store I want to do a search over possible architectures


In [17]:
# Add project root to path so we can import from data_manipulation and model
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    if (cwd / "data_manipulation").is_dir():
        return cwd
    if (cwd.parent / "data_manipulation").is_dir():
        return cwd.parent
    return cwd  # fallback

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [18]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from functools import partial

# Imports from data_manipulation and model
from data_manipulation.data_split import create_dataloader, DemandDataset
from model.functions import pinball_loss, rmse, train, get_test_loss

In [19]:
class simple_net(nn.Module):
    def __init__(self, layers:list[int]):
        super().__init__()
        
        self.net = nn.Sequential()

        # Add layers
        for i, layer in enumerate(layers[:-2]):
            self.net.add_module(f"fc{i}", nn.Linear(layer, layers[i+1]))
            if i < len(layers) - 1:
                self.net.add_module(f"relu{i}", nn.ReLU())

        # Add last layer
        self.net.add_module(f"fc{len(layers)-1}", nn.Linear(layers[-2], layers[-1]))
        

    def forward(self, x):
        return self.net(x)

In [20]:
all_specs = [
    "sales",
    "7_day_rolling_ema",	
    "30_day_rolling_ema",
    "90_day_rolling_ema",
    "30_day_rolling_min",
    "5_day_lag",
    "6_day_lag",
    "7_day_lag",
    "dif_180_day",
]

In [21]:
h_cost = 1
l_cost = 3

num_epochs = 500

In [22]:
in_features = 8
out_features = 1

layer_combos =[
    [in_features, 80, 80, out_features],
    [in_features, 80, 80, 80, out_features],
    [in_features, 90, 90, out_features],
    [in_features, 90, 90, 90, out_features],
    [in_features, 100, 100, out_features],
    [in_features, 100, 100, 100, out_features],
    [in_features, 110, 110, out_features],
    [in_features, 110, 110, 110, out_features],
    [in_features, 120, 120, out_features],
    [in_features, 120, 120, 120, out_features],
    [in_features, 130, 130, out_features],
    [in_features, 130, 130, 130, out_features],
]

In [23]:
from collections import defaultdict

losses = defaultdict(list)

# Number of runs
for i in range(10):
    # get shuffled data
    train_loader, val_loader, test_loader = create_dataloader(
        batch_size=8, 
        test_batch_size=1,
        pin_memory=False,
        data_mask=[
            ("item", 1),
            ("store", 1)
        ],
        specs=all_specs,
        )

    for layer_combo in layer_combos:
        net_1 = simple_net(layer_combo)
        loss = partial(pinball_loss, h_cost=h_cost, l_cost=l_cost)
        optimizer = torch.optim.Adam(net_1.parameters(), lr=0.001)
        _, _ = train(net_1, optimizer, loss, train_loader, val_loader, epochs=200, eval_interval=10, device="cpu", use_tqdm=False)
        test_loss = get_test_loss(net_1, test_loader, loss, "cpu")
        
        losses[str(layer_combo)].append(torch.tensor(test_loss, dtype=torch.float32).mean().item())

        # Print average loss
        print(f"Test Loss for {str(layer_combo):<50}: {torch.tensor(test_loss, dtype=torch.float32).mean().item():.4f}")


Test Loss for [8, 80, 80, 1]                                    : 7.7356
Test Loss for [8, 80, 80, 80, 1]                                : 7.4725
Test Loss for [8, 90, 90, 1]                                    : 8.0135
Test Loss for [8, 90, 90, 90, 1]                                : 7.8924
Test Loss for [8, 100, 100, 1]                                  : 7.5570
Test Loss for [8, 100, 100, 100, 1]                             : 8.2000
Test Loss for [8, 110, 110, 1]                                  : 7.4990
Test Loss for [8, 110, 110, 110, 1]                             : 7.8968
Test Loss for [8, 120, 120, 1]                                  : 7.3747
Test Loss for [8, 120, 120, 120, 1]                             : 7.6337
Test Loss for [8, 130, 130, 1]                                  : 8.3579
Test Loss for [8, 130, 130, 130, 1]                             : 7.7633
Test Loss for [8, 80, 80, 1]                                    : 7.8595
Test Loss for [8, 80, 80, 80, 1]                   

In [24]:
# Print the sorted lowest to highest average loss
for layer_combo, losses in sorted(losses.items(), key=lambda x: torch.tensor(x[1]).mean()):
    print(f"Layer Combo: {layer_combo}, Average Loss: {torch.tensor(losses).mean():.4f}")


Layer Combo: [8, 120, 120, 1], Average Loss: 7.4579
Layer Combo: [8, 80, 80, 80, 1], Average Loss: 7.4905
Layer Combo: [8, 100, 100, 1], Average Loss: 7.6204
Layer Combo: [8, 110, 110, 1], Average Loss: 7.6359
Layer Combo: [8, 90, 90, 90, 1], Average Loss: 7.6801
Layer Combo: [8, 90, 90, 1], Average Loss: 7.6955
Layer Combo: [8, 80, 80, 1], Average Loss: 7.7489
Layer Combo: [8, 120, 120, 120, 1], Average Loss: 7.8859
Layer Combo: [8, 130, 130, 130, 1], Average Loss: 7.8996
Layer Combo: [8, 100, 100, 100, 1], Average Loss: 7.9166
Layer Combo: [8, 130, 130, 1], Average Loss: 8.0189
Layer Combo: [8, 110, 110, 110, 1], Average Loss: 8.1104
